In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd
import numpy as np
import os

from astropy.time import Time

In [ ]:
# set the directory path
directory_path = 'epoch_9/'

# create an empty list to store dataframes
dfs = []

# loop through all files in the directory that start with "beam_inf"
for filename in os.listdir(directory_path):
    if filename.startswith('beam_inf') and filename.endswith('.csv'):
        # read the CSV file into a pandas dataframe
        filepath = os.path.join(directory_path, filename)
        df = pd.read_csv(filepath)
        # append the dataframe to the list
        dfs.append(df)
        
# concatenate all dataframes into a single dataframe
combined_df = pd.concat(dfs, ignore_index=True)


In [ ]:
# plot the beam time of each dataframe
plt.figure(figsize=(20, 5))
plt.scatter(combined_df['BEAM_TIME'], combined_df['BEAM_NUM'])
plt.title('Beam Time')
plt.xlabel('Time')
plt.ylabel('Beam Number')
plt.show()

In [ ]:
# create an empty list to store the max and min values
max_min_values = []

# loop through all dataframes in the list
for df in dfs:
    # find the max and min values of RA_DEG and DEC_DEG
    max_ra = df['RA_DEG'].max()
    min_ra = df['RA_DEG'].min()
    max_dec = df['DEC_DEG'].max()
    min_dec = df['DEC_DEG'].min()
    diff_ra = max_ra - min_ra
    diff_dec = max_dec - min_dec
    # append the max and min values to the list
    max_min_values.append({'max_ra': max_ra, 'min_ra': min_ra, 'max_dec': max_dec, 'min_dec': min_dec,
                           'diff_ra': diff_ra, 'diff_dec': diff_dec})

# create a new dataframe with the max and min values
max_min_df = pd.DataFrame(max_min_values)
print(max_min_df)


In [ ]:
# create an empty list to store the max and min values
beam_time_values = []

# loop through all dataframes in the list
for df in dfs:
    # find the max and min values of RA_DEG and DEC_DEG
    max_time = df['BEAM_TIME'].max()
    min_time = df['BEAM_TIME'].min()
    diff_time = max_time - min_time
    # append the max and min values to the list
    beam_time_values.append({'max_time': max_time, 'min_time': min_time, 'diff_time': diff_time})

# create a new dataframe with the max and min values
beam_time_df = pd.DataFrame(beam_time_values)
print(beam_time_df)

In [ ]:
new_df = combined_df[combined_df['BEAM_TIME'].between(5.140e9, 5.145e9)]

plt.figure(figsize=(20, 5))
plt.scatter(new_df['BEAM_TIME'], new_df['BEAM_NUM'])
plt.title('Beam Time')
plt.xlabel('Time')
plt.ylabel('Beam Number')
plt.show()

## RACSLow3 Fields

In [ ]:
# RACSLow3 fields
df_field_data = pd.read_csv('epoch_9/field_data.csv')
print(df_field_data['SCAN_START'])

In [ ]:
# remove all the invalid entries from the dataframe
df_field_data = df_field_data[df_field_data['SCAN_START'] != -1]
df_field_data = df_field_data[df_field_data['COMMENT'].isnull()]
df_field_data = df_field_data[df_field_data['SCAN_LEN'] > 800]


# plot the scan start time against the scan time length for each scan
plt.figure(figsize=(20, 5))
plt.scatter(df_field_data['SCAN_START'], df_field_data['SCAN_LEN'])
plt.title('Scan Time')
plt.xlabel('Scan Start Time')
plt.ylabel('Scan Time Length')
plt.show()

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_field_data['RA_DEG'], df_field_data['DEC_DEG'], c=df_field_data['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()


## MJD to UTC Function

In [ ]:
def mjd2utc(mjd_seconds):
    # create a Time object with the MJD seconds
    t = Time(mjd_seconds/86400, format='mjd', scale='utc')

    # convert the time to YYYYMMDD HH:MM:SS format
    time_str = t.datetime.strftime('%Y-%m-%d %H:%M:%S')
    fin_time = time_str + '.' + str(mjd_seconds).split('.')[1]
    return fin_time


## FIRST Field

In [ ]:
# create a new dataframe with fields that already overlap with those mapped by FIRST
df_req_fields1 = df_field_data[((df_field_data['RA_DEG'].between(135, 240)) & (df_field_data['DEC_DEG'].between(-10, 30)))]
df_req_fields2 = df_field_data[(((df_field_data['RA_DEG'] < 45) | (df_field_data['RA_DEG'] > 315)) & (df_field_data['DEC_DEG'].between(-10, 10)))]
df_req_fields_first = pd.concat([df_req_fields1, df_req_fields2], ignore_index=True)

In [ ]:
# create a new dataframe with fields that already overlap with those mapped by FIRST
df_req_fields1 = df_field_data[((df_field_data['RA_DEG'].between(135, 240)) & (df_field_data['DEC_DEG'].between(-10, 15)))]
df_req_fields2 = df_field_data[(((df_field_data['RA_DEG'] < 45) | (df_field_data['RA_DEG'] > 315)) & (df_field_data['DEC_DEG'].between(-10, 10)))]
df_req_fields_first = pd.concat([df_req_fields1, df_req_fields2], ignore_index=True)
df_req_fields_first = df_req_fields_first[df_req_fields_first['SCAN_LEN'].between(880, 940)]

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_req_fields_first['RA_DEG'], df_req_fields_first['DEC_DEG'], c=df_req_fields_first['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
plt.ylim(-40, 50)
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()

In [ ]:
# save the field names to a numpy array for use in the crossmatch notebook
field_list = df_req_fields_first['FIELD_NAME'].tolist()
# np.save('RACSLow_Fields.npy', field_list)

sbid_list = df_req_fields_first['SBID'].tolist()
# np.save('RACSLow_SBIDs.npy', sbid_list)

cal_sbid_list = df_req_fields_first['CAL_SBID'].tolist()
# np.save('RACSLow_CAL_SBIDs.npy', cal_sbid_list)

df_req_fields_first['UTC_SCAN_START'] = df_req_fields_first['SCAN_START'].apply(mjd2utc)
time_list = df_req_fields_first['UTC_SCAN_START'].tolist()
# np.save('RACSLow_Times.npy', time_list)

# print(field_list)

# Create a new dataframe
df_list_first = pd.DataFrame()

# Assign values to the columns
df_list_first['Field Name'] = field_list
df_list_first['SBID'] = sbid_list
df_list_first['CAL_SBID'] = cal_sbid_list
df_list_first['UTC Scan Start Time'] = time_list

df_list_first.sort_values('UTC Scan Start Time', inplace=True)
df_list_first.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
# Save the field names to a numpy array and then to a numpy file
np.save('RACSLow3_FIRST_CELEBI_Testing.npy', np.array(df_list_first))

## VLASS Field

In [ ]:
# create a new dataframe with fields that already overlap with those mapped by VLASS
df_req_fields_vlass = df_field_data[(df_field_data['DEC_DEG'].between(-40, 50))]

In [ ]:
# create a new dataframe with fields that already overlap with those mapped by VLASS but excluding fields above DEC +30 deg and galactic plane
df_req_fields_vlass_dec = df_req_fields_vlass[df_req_fields_vlass['DEC_DEG'] < 30]
df_req_fields_vlass_gal = df_req_fields_vlass_dec[(df_req_fields_vlass_dec['GAL_LAT'] > 11) | (df_req_fields_vlass_dec['GAL_LAT'] < -11)]

In [ ]:
# create a new dataframe with fields that already overlap with those mapped outside VLASS
df_req_fields_out_vlass = df_field_data[(df_field_data['DEC_DEG'].between(-90, -40))]

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_req_fields_out_vlass['RA_DEG'], df_req_fields_out_vlass['DEC_DEG'], c=df_req_fields_out_vlass['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
plt.ylim(-90, 50)
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_req_fields_vlass_gal['RA_DEG'], df_req_fields_vlass_gal['DEC_DEG'], c=df_req_fields_vlass_gal['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
plt.ylim(-90, 50)
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()

In [ ]:
# save the field names to a numpy array for use in the crossmatch notebook
field_list = df_req_fields_out_vlass['FIELD_NAME'].tolist()
# np.save('RACSLow_Fields.npy', field_list)

sbid_list = df_req_fields_out_vlass['SBID'].tolist()
# np.save('RACSLow_SBIDs.npy', sbid_list)

cal_sbid_list = df_req_fields_out_vlass['CAL_SBID'].tolist()
# np.save('RACSLow_CAL_SBIDs.npy', cal_sbid_list)

df_req_fields_out_vlass['UTC_SCAN_START'] = df_req_fields_out_vlass['SCAN_START'].apply(mjd2utc)
time_list = df_req_fields_out_vlass['UTC_SCAN_START'].tolist()
# np.save('RACSLow_Times.npy', time_list)

# print(field_list)

# Create a new dataframe
df_list_vlass = pd.DataFrame()

# Assign values to the columns
df_list_vlass['Field Name'] = field_list
df_list_vlass['SBID'] = sbid_list
df_list_vlass['CAL_SBID'] = cal_sbid_list
df_list_vlass['UTC Scan Start Time'] = time_list

df_list_vlass.sort_values('UTC Scan Start Time', inplace=True)
df_list_vlass.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
# Save the field names to a numpy array and then to a numpy file
np.save('RACSLow3_outsideVLASS.npy', np.array(df_list_vlass))

## RACSLow3 without DEC +30 and Galactic PLane

In [ ]:
# create a new dataframe with fields around the galactic region and gaalctic cut separations of the catalogue
df_req_fields_dec = df_field_data[df_field_data['DEC_DEG'] < 30]
df_req_fields_gal = df_req_fields_dec[(df_field_data['GAL_LAT'] > 11) | (df_field_data['GAL_LAT'] < -11)]

In [ ]:
plt.figure(figsize=(10, 8))
plt.subplot(111, projection="mollweide")
plt.scatter(np.radians(df_req_fields_gal['GAL_LONG'])-np.pi, np.radians(df_req_fields_gal['GAL_LAT']), c=df_req_fields_gal['SCAN_START'], cmap='jet')
plt.title('Field vs Scan Start Time')
plt.xlabel('Galactic Longitude (in degrees)')
plt.ylabel('Galactic Latitude (in degrees)')
clb = plt.colorbar()
clb.set_label('Scan Start Time (in seconds)')
plt.grid(True)
plt.show()

In [ ]:
# plot the field vs the scan observation time
plt.figure(figsize=(10, 8))
plt.scatter(df_req_fields_gal['RA_DEG'], df_req_fields_gal['DEC_DEG'], c=df_req_fields_gal['SCAN_LEN'], cmap='jet')
plt.title('Field vs Observation Time')
plt.xlabel('RA (in degrees)')
plt.ylabel('DEC (in degrees)')
clb = plt.colorbar()
clb.set_label('Scan Length (in seconds)')
plt.show()

# For full RACSLow3 coverage

In [ ]:
# save the field names to a numpy array for use in the crossmatch notebook
field_list = df_field_data['FIELD_NAME'].tolist()
# np.save('RACSLow_Fields.npy', field_list)

sbid_list = df_field_data['SBID'].tolist()
# np.save('RACSLow_SBIDs.npy', sbid_list)

cal_sbid_list = df_field_data['CAL_SBID'].tolist()
# np.save('RACSLow_CAL_SBIDs.npy', cal_sbid_list)

df_field_data['UTC_SCAN_START'] = df_field_data['SCAN_START'].apply(mjd2utc)
time_list = df_field_data['UTC_SCAN_START'].tolist()
# np.save('RACSLow_Times.npy', time_list)

# print(field_list)

# Create a new dataframe
df_list = pd.DataFrame()

# Assign values to the columns
df_list['Field Name'] = field_list
df_list['SBID'] = sbid_list
df_list['CAL_SBID'] = cal_sbid_list
df_list['UTC Scan Start Time'] = time_list

df_list.sort_values('UTC Scan Start Time', inplace=True)
df_list.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
sbid = [str(df_list.iloc[i]['SBID']) for i in range(len(df_list))]
cal_sbid = [str(df_list.iloc[i]['CAL_SBID']) for i in range(len(df_list))]

cal_sbid_counts = Counter(cal_sbid)

cal_sbids = list(cal_sbid_counts.keys())
counts = list(cal_sbid_counts.values())

cal_sbid_to_sbid = {}

for i in range(len(df_list)):
    if cal_sbid[i] in cal_sbid_to_sbid:
        if sbid[i] not in cal_sbid_to_sbid[cal_sbid[i]]:
            cal_sbid_to_sbid[cal_sbid[i]].append(sbid[i])
    else:
        cal_sbid_to_sbid[cal_sbid[i]] = [sbid[i]]

# Print the corresponding SBID values for each CAL_SBID value
for cal_sbid_value, sbid_values in cal_sbid_to_sbid.items():
    print(f"CAL_SBID: {cal_sbid_value}, SBID values: {', '.join(sbid_values)}")

sbid_range = [f"{min(cal_sbid_to_sbid[cal_sbids[i]])}-{max(cal_sbid_to_sbid[cal_sbids[i]])}" for i in range(len(cal_sbids))]

plt.figure(figsize=(15, 6))
plt.bar(cal_sbids, counts)
plt.xlabel('SBID Range')
plt.ylabel('Number of Scans')
plt.title('SBID vs Number of Scans')
# Print the text at the bottom of the histogram
for i in range(len(sbid_range)):
    plt.text(i, counts[i], cal_sbids[i], ha='center', va='bottom', fontsize=8, rotation=90, mouseover=cal_sbids[i])

plt.xticks(range(len(sbid_range)), sbid_range, rotation=90)
plt.ylim(top=max(counts)+15)
plt.show()


In [ ]:
# Save the field names to a numpy array for use in the crossmatch notebook
# Convert df_list to a numpy array
df_list_array = np.array(df_list)

# Save the numpy array to a file
np.save('RACSLow3_FullList.npy', df_list_array)


In [ ]:
date = [str(df_list.iloc[i]['UTC Scan Start Time'])[0:10] for i in range(len(df_list))]

date_counts = Counter(date)

for date, count in date_counts.items():
    print(f"{date}: {count}")

dates = list(date_counts.keys())
counts = list(date_counts.values())

plt.figure(figsize=(15, 6))
plt.bar(dates, counts)
plt.xlabel('Date')
plt.ylabel('Number of Scans')
plt.title('Date vs Number of Scans')
plt.xticks(rotation=90)
plt.show()



In [ ]:
# save the field names to a numpy array for use in the crossmatch notebook
field_list = df_req_fields_gal['FIELD_NAME'].tolist()
# np.save('RACSLow_Fields.npy', field_list)

sbid_list = df_req_fields_gal['SBID'].tolist()
# np.save('RACSLow_SBIDs.npy', sbid_list)

cal_sbid_list = df_req_fields_gal['CAL_SBID'].tolist()
# np.save('RACSLow_CAL_SBIDs.npy', cal_sbid_list)

df_req_fields_gal['UTC_SCAN_START'] = df_req_fields_gal['SCAN_START'].apply(mjd2utc)
time_list = df_req_fields_gal['UTC_SCAN_START'].tolist()
# np.save('RACSLow_Times.npy', time_list)

# print(field_list)

# Create a new dataframe
df_list_gal = pd.DataFrame()

# Assign values to the columns
df_list_gal['Field Name'] = field_list
df_list_gal['SBID'] = sbid_list
df_list_gal['CAL_SBID'] = cal_sbid_list
df_list_gal['UTC Scan Start Time'] = time_list

df_list_gal.sort_values('UTC Scan Start Time', inplace=True)
df_list_gal.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
# Save the field names to a numpy array for use in the crossmatch notebook
# Convert df_list to a numpy array
df_list_array_gal = np.array(df_list_gal)

# Save the numpy array to a file
np.save('RACSLow3_GalList.npy', df_list_array_gal)

In [ ]:
# Create a new dataframe with the RACSLow field names and their corresponding SBID and UTC Scan Start Time
df_list = pd.DataFrame(np.load('RACSLow3_FullList.npy', allow_pickle=True), columns=['Field Name', 'SBID', 'CAL_SBID', 'UTC Scan Start Time'])

df_list.sort_values('UTC Scan Start Time', inplace=True)
df_list.drop_duplicates(subset='Field Name', keep='last', inplace=True)

# Create a new dataframe with the RACSLow field names and their corresponding SBID and UTC Scan Start Time
df_list2 = pd.DataFrame(np.load('RACSLow3_GalList.npy', allow_pickle=True), columns=['Field Name', 'SBID', 'CAL_SBID', 'UTC Scan Start Time'])

df_list2.sort_values('UTC Scan Start Time', inplace=True)
df_list2.drop_duplicates(subset='Field Name', keep='last', inplace=True)

In [ ]:
# Set the reading directory path for the RACSLow beam information
directory_path = 'epoch_9/'

# Create the saving directory if it doesn't exist
directory_main = 'D:\ASKAP Astrometry Storage'

directory = os.path.join(directory_main, 'RACSLow3_Queries')
os.makedirs(directory, exist_ok=True)

# Create an empty list to store RACSLow scan information
df_racs = []

for i in range(len(df_list)):
    filename = f'beam_inf_{df_list["SBID"].values[i]}-{df_list["Field Name"].values[i]}.csv'
    filepath = os.path.join(directory_path, filename)
    
    # Read the RACS beam information to get beam number and beam center
    df = pd.read_csv(filepath)
    
    df['FIELD_NAME'] = filename.split('.')[0][-12:]
    df['SBID'] = int(filename.split('.')[0].split('beam_inf_')[1][:-13])
    df['CAL_SBID'] = df_list[df_list['Field Name'] == filename.split('.')[0][-12:]]['CAL_SBID'].values[0]
    df['UTC_SCAN_START'] = df_list[df_list['Field Name'] == filename.split('.')[0][-12:]]['UTC Scan Start Time'].values[0]
    
    # Append the dataframe to the list
    df_racs.append(df)

In [ ]:
indices = [index for index, row in df_list.iterrows() if row['Field Name'] not in df_list2['Field Name'].values]
indices, len(indices)


In [ ]:
1493-1052